# SQL Analysis — Olist E-Commerce Dataset

**Objective:** Demonstrate SQL proficiency on the Olist dataset using SQLite — covering JOINs, GROUP BY, Window Functions, CTEs, Subqueries, and CASE statements.

**Approach:** Load cleaned CSV files into an in-memory SQLite database and run analytical queries.

**Queries Covered:**
1. Basic Aggregations & GROUP BY
2. Multi-table JOINs
3. Window Functions (RANK, ROW_NUMBER, LAG, Running Totals)
4. Common Table Expressions (CTEs)
5. Subqueries & Nested Queries
6. CASE Statements & Conditional Logic
7. Date Functions & Time-based Analysis
8. Advanced Business Queries

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

# Load all cleaned CSVs into SQLite tables
tables = {
    "orders": "cleaned/orders_cleaned.csv",
    "customers": "cleaned/customers_cleaned.csv",
    "items": "cleaned/items_cleaned.csv",
    "products": "cleaned/products_cleaned.csv",
    "payments": "cleaned/payments_cleaned.csv",
    "reviews": "cleaned/reviews_cleaned.csv",
    "sellers": "cleaned/sellers_cleaned.csv",
    "geolocation": "cleaned/geolocation_cleaned.csv",
}

for table_name, file_path in tables.items():
    df = pd.read_csv(file_path)
    df.to_sql(table_name, conn, index=False, if_exists="replace")
    print(f"Loaded {table_name}: {len(df):,} rows")

def sql(query):
    """Helper to run SQL and return a DataFrame"""
    return pd.read_sql_query(query, conn)

print("\nAll tables loaded into SQLite. Ready for queries.")

## 1. Basic Aggregations & GROUP BY

In [ ]:
# Q1: Total orders, revenue, and avg order value
sql("""
SELECT 
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(i.total_price), 2) AS total_revenue,
    ROUND(AVG(i.total_price), 2) AS avg_item_value,
    COUNT(DISTINCT o.customer_id) AS total_customers
FROM orders o
JOIN items i ON o.order_id = i.order_id
""")

In [ ]:
# Q2: Revenue by state — Top 10
sql("""
SELECT 
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(i.total_price), 2) AS total_revenue,
    ROUND(AVG(i.total_price), 2) AS avg_order_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN items i ON o.order_id = i.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10
""")

In [ ]:
# Q3: Order count by status
sql("""
SELECT 
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS percentage
FROM orders
GROUP BY order_status
ORDER BY order_count DESC
""")

In [ ]:
# Q4: Top 10 product categories by revenue
sql("""
SELECT 
    p.category,
    COUNT(DISTINCT i.order_id) AS total_orders,
    ROUND(SUM(i.total_price), 2) AS total_revenue,
    ROUND(AVG(i.price), 2) AS avg_price
FROM items i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
LIMIT 10
""")

## 2. Multi-Table JOINs

In [ ]:
# Q5: Full order details — 5-table JOIN (orders + customers + items + products + payments)
sql("""
SELECT 
    o.order_id,
    c.customer_city,
    c.customer_state,
    p.category,
    i.price,
    i.freight_value,
    pay.total_payment,
    pay.payment_type,
    o.order_status,
    o.order_purchase_timestamp
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN items i ON o.order_id = i.order_id
JOIN products p ON i.product_id = p.product_id
JOIN payments pay ON o.order_id = pay.order_id
LIMIT 10
""")

In [ ]:
# Q6: Avg review score per seller — JOIN orders + items + sellers + reviews
sql("""
SELECT 
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT i.order_id) AS total_orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    ROUND(SUM(i.total_price), 2) AS total_revenue
FROM items i
JOIN sellers s ON i.seller_id = s.seller_id
JOIN reviews r ON i.order_id = r.order_id
GROUP BY s.seller_city, s.seller_state
HAVING total_orders >= 50
ORDER BY avg_review_score ASC
LIMIT 10
""")

## 3. Window Functions (RANK, ROW_NUMBER, LAG, Running Totals)

In [ ]:
# Q7: Rank states by revenue using RANK()
sql("""
SELECT 
    customer_state,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    ROUND(total_revenue * 100.0 / SUM(total_revenue) OVER (), 2) AS revenue_pct
FROM (
    SELECT 
        c.customer_state,
        ROUND(SUM(i.total_price), 2) AS total_revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN items i ON o.order_id = i.order_id
    GROUP BY c.customer_state
)
ORDER BY revenue_rank
LIMIT 10
""")

In [ ]:
# Q8: Monthly revenue with running total and MoM growth using LAG()
sql("""
SELECT 
    order_month,
    monthly_revenue,
    SUM(monthly_revenue) OVER (ORDER BY order_month) AS running_total,
    LAG(monthly_revenue) OVER (ORDER BY order_month) AS prev_month_revenue,
    ROUND(
        (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY order_month)) * 100.0 
        / LAG(monthly_revenue) OVER (ORDER BY order_month), 2
    ) AS mom_growth_pct
FROM (
    SELECT 
        order_month,
        ROUND(SUM(i.total_price), 2) AS monthly_revenue
    FROM orders o
    JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY order_month
)
ORDER BY order_month
""")

In [ ]:
# Q9: Top 3 categories per state using ROW_NUMBER() — PARTITION BY
sql("""
SELECT * FROM (
    SELECT 
        c.customer_state,
        p.category,
        ROUND(SUM(i.total_price), 2) AS category_revenue,
        ROW_NUMBER() OVER (PARTITION BY c.customer_state ORDER BY SUM(i.total_price) DESC) AS rn
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN items i ON o.order_id = i.order_id
    JOIN products p ON i.product_id = p.product_id
    GROUP BY c.customer_state, p.category
)
WHERE rn <= 3
ORDER BY customer_state, rn
LIMIT 15
""")

In [ ]:
# Q10: Cumulative percentage of revenue by category (Pareto analysis)
sql("""
SELECT 
    category,
    total_revenue,
    SUM(total_revenue) OVER (ORDER BY total_revenue DESC) AS cumulative_revenue,
    ROUND(
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC) * 100.0 
        / SUM(total_revenue) OVER (), 2
    ) AS cumulative_pct
FROM (
    SELECT 
        p.category,
        ROUND(SUM(i.total_price), 2) AS total_revenue
    FROM items i
    JOIN products p ON i.product_id = p.product_id
    GROUP BY p.category
)
ORDER BY total_revenue DESC
LIMIT 15
""")

## 4. Common Table Expressions (CTEs)

In [ ]:
# Q11: RFM Analysis using CTEs
sql("""
WITH customer_orders AS (
    SELECT 
        c.customer_unique_id,
        MAX(o.order_purchase_timestamp) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(i.total_price), 2) AS monetary
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
),
rfm AS (
    SELECT 
        customer_unique_id,
        CAST(julianday('2018-10-18') - julianday(last_order_date) AS INTEGER) AS recency,
        frequency,
        monetary
    FROM customer_orders
)
SELECT 
    CASE 
        WHEN recency <= 30 THEN '0-30 days'
        WHEN recency <= 90 THEN '31-90 days'
        WHEN recency <= 180 THEN '91-180 days'
        WHEN recency <= 365 THEN '181-365 days'
        ELSE '365+ days'
    END AS recency_bucket,
    COUNT(*) AS customer_count,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary
FROM rfm
GROUP BY recency_bucket
ORDER BY MIN(recency)
""")

In [ ]:
# Q12: Late delivery impact on reviews — Multi-CTE
sql("""
WITH delivery_data AS (
    SELECT 
        order_id,
        CAST(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) AS INTEGER) AS delay_days,
        CAST(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp) AS INTEGER) AS delivery_days
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date != 'Not Delivered'
),
delivery_reviews AS (
    SELECT 
        d.order_id,
        d.delay_days,
        d.delivery_days,
        r.review_score,
        CASE WHEN d.delay_days > 0 THEN 'Late' ELSE 'On Time' END AS delivery_status
    FROM delivery_data d
    JOIN reviews r ON d.order_id = r.order_id
)
SELECT 
    delivery_status,
    COUNT(*) AS total_orders,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
    ROUND(AVG(delay_days), 1) AS avg_delay_days,
    SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) AS one_star_count,
    ROUND(SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS one_star_pct
FROM delivery_reviews
GROUP BY delivery_status
""")

In [ ]:
# Q13: Seller performance analysis — CTE with multiple metrics
sql("""
WITH seller_metrics AS (
    SELECT 
        i.seller_id,
        s.seller_city,
        s.seller_state,
        COUNT(DISTINCT i.order_id) AS total_orders,
        ROUND(SUM(i.total_price), 2) AS total_revenue,
        ROUND(AVG(i.price), 2) AS avg_price
    FROM items i
    JOIN sellers s ON i.seller_id = s.seller_id
    GROUP BY i.seller_id, s.seller_city, s.seller_state
),
seller_reviews AS (
    SELECT 
        i.seller_id,
        ROUND(AVG(r.review_score), 2) AS avg_review
    FROM items i
    JOIN reviews r ON i.order_id = r.order_id
    GROUP BY i.seller_id
)
SELECT 
    sm.seller_city,
    sm.seller_state,
    sm.total_orders,
    sm.total_revenue,
    sm.avg_price,
    sr.avg_review,
    RANK() OVER (ORDER BY sm.total_revenue DESC) AS revenue_rank
FROM seller_metrics sm
JOIN seller_reviews sr ON sm.seller_id = sr.seller_id
WHERE sm.total_orders >= 20
ORDER BY sm.total_revenue DESC
LIMIT 15
""")

## 5. Subqueries & Nested Queries

In [ ]:
# Q14: Orders with above-average payment value (correlated subquery)
sql("""
SELECT 
    o.order_id,
    c.customer_city,
    c.customer_state,
    p.total_payment,
    r.review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE p.total_payment > (SELECT AVG(total_payment) FROM payments)
ORDER BY p.total_payment DESC
LIMIT 15
""")

In [ ]:
# Q15: States where avg delivery delay is worse than the national average
sql("""
SELECT 
    c.customer_state,
    COUNT(*) AS total_delivered,
    ROUND(AVG(o.delivery_delay_days), 2) AS avg_delay_days,
    ROUND((SELECT AVG(delivery_delay_days) FROM orders WHERE order_status = 'delivered'), 2) AS national_avg_delay,
    ROUND(AVG(o.delivery_delay_days) - (SELECT AVG(delivery_delay_days) FROM orders WHERE order_status = 'delivered'), 2) AS vs_national
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING avg_delay_days > (SELECT AVG(delivery_delay_days) FROM orders WHERE order_status = 'delivered')
ORDER BY avg_delay_days DESC
""")

## 6. CASE Statements & Conditional Logic

In [ ]:
# Q16: Customer segmentation by spend tier using CASE
sql("""
WITH customer_spend AS (
    SELECT 
        c.customer_unique_id,
        ROUND(SUM(i.total_price), 2) AS total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN items i ON o.order_id = i.order_id
    GROUP BY c.customer_unique_id
)
SELECT 
    CASE 
        WHEN total_spend >= 500 THEN 'High Spender (500+)'
        WHEN total_spend >= 200 THEN 'Medium Spender (200-500)'
        WHEN total_spend >= 100 THEN 'Low Spender (100-200)'
        ELSE 'Micro Spender (<100)'
    END AS spend_tier,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_spend), 2) AS avg_spend,
    ROUND(MIN(total_spend), 2) AS min_spend,
    ROUND(MAX(total_spend), 2) AS max_spend
FROM customer_spend
GROUP BY spend_tier
ORDER BY avg_spend DESC
""")

In [ ]:
# Q17: Review sentiment breakdown with delivery context
sql("""
SELECT 
    CASE 
        WHEN r.review_score >= 4 THEN 'Positive (4-5)'
        WHEN r.review_score = 3 THEN 'Neutral (3)'
        ELSE 'Negative (1-2)'
    END AS sentiment,
    COUNT(*) AS review_count,
    ROUND(AVG(o.delivery_delay_days), 2) AS avg_delay_days,
    ROUND(AVG(o.actual_delivery_days), 2) AS avg_delivery_days,
    SUM(CASE WHEN o.is_late = 1 THEN 1 ELSE 0 END) AS late_deliveries,
    ROUND(SUM(CASE WHEN o.is_late = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS late_pct
FROM reviews r
JOIN orders o ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY sentiment
ORDER BY avg_delay_days DESC
""")

## 7. Date Functions & Time-Based Analysis

In [ ]:
# Q18: Hourly order distribution — when do customers shop?
sql("""
SELECT 
    CAST(strftime('%H', order_purchase_timestamp) AS INTEGER) AS hour_of_day,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct_of_total
FROM orders
GROUP BY hour_of_day
ORDER BY hour_of_day
""")

In [ ]:
# Q19: Quarter-over-quarter revenue comparison
sql("""
WITH quarterly AS (
    SELECT 
        strftime('%Y', o.order_purchase_timestamp) AS year,
        CASE 
            WHEN CAST(strftime('%m', o.order_purchase_timestamp) AS INTEGER) BETWEEN 1 AND 3 THEN 'Q1'
            WHEN CAST(strftime('%m', o.order_purchase_timestamp) AS INTEGER) BETWEEN 4 AND 6 THEN 'Q2'
            WHEN CAST(strftime('%m', o.order_purchase_timestamp) AS INTEGER) BETWEEN 7 AND 9 THEN 'Q3'
            ELSE 'Q4'
        END AS quarter,
        ROUND(SUM(i.total_price), 2) AS revenue
    FROM orders o
    JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY year, quarter
)
SELECT 
    year,
    quarter,
    year || '-' || quarter AS period,
    revenue,
    LAG(revenue) OVER (ORDER BY year, quarter) AS prev_quarter_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY year, quarter)) * 100.0 
        / LAG(revenue) OVER (ORDER BY year, quarter), 2
    ) AS qoq_growth_pct
FROM quarterly
ORDER BY year, quarter
""")

In [ ]:
# Q20: Average days between order and approval — processing time analysis
sql("""
SELECT 
    order_day_of_week,
    COUNT(*) AS total_orders,
    ROUND(AVG(
        CASE 
            WHEN order_approved_at != order_purchase_timestamp 
            THEN julianday(order_approved_at) - julianday(order_purchase_timestamp)
            ELSE 0
        END
    ), 2) AS avg_approval_hours_in_days,
    ROUND(AVG(actual_delivery_days), 1) AS avg_delivery_days
FROM orders
WHERE order_status = 'delivered'
GROUP BY order_day_of_week
ORDER BY 
    CASE order_day_of_week
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END
""")

## 8. Advanced Business Queries

In [ ]:
# Q21: Repeat customers vs one-time buyers
sql("""
WITH customer_frequency AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count,
        ROUND(SUM(i.total_price), 2) AS total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT 
    CASE 
        WHEN order_count = 1 THEN 'One-time Buyer'
        WHEN order_count = 2 THEN 'Two-time Buyer'
        WHEN order_count >= 3 THEN 'Repeat Buyer (3+)'
    END AS customer_type,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_frequency), 2) AS pct,
    ROUND(AVG(total_spend), 2) AS avg_lifetime_value,
    ROUND(SUM(total_spend), 2) AS total_revenue_contribution
FROM customer_frequency
GROUP BY customer_type
ORDER BY customer_count DESC
""")

In [ ]:
# Q22: Freight cost as percentage of order value — by category
sql("""
SELECT 
    p.category,
    COUNT(*) AS total_items,
    ROUND(AVG(i.price), 2) AS avg_price,
    ROUND(AVG(i.freight_value), 2) AS avg_freight,
    ROUND(AVG(i.freight_value) * 100.0 / AVG(i.price), 2) AS freight_pct_of_price
FROM items i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.category
HAVING total_items >= 100
ORDER BY freight_pct_of_price DESC
LIMIT 15
""")

In [ ]:
# Q23: Payment installment analysis — do more installments = higher spend?
sql("""
SELECT 
    pay.payment_installments,
    COUNT(*) AS order_count,
    ROUND(AVG(pay.total_payment), 2) AS avg_payment,
    ROUND(AVG(r.review_score), 2) AS avg_review,
    pay.payment_type AS most_common_type
FROM payments pay
JOIN reviews r ON pay.order_id = r.order_id
WHERE pay.payment_installments BETWEEN 1 AND 12
GROUP BY pay.payment_installments
ORDER BY pay.payment_installments
""")

In [ ]:
# Q24: Cross-state seller-customer analysis — where do sellers ship to?
sql("""
SELECT 
    s.seller_state AS seller_from,
    c.customer_state AS customer_to,
    COUNT(DISTINCT i.order_id) AS total_orders,
    ROUND(SUM(i.total_price), 2) AS total_revenue,
    ROUND(AVG(i.freight_value), 2) AS avg_freight,
    CASE WHEN s.seller_state = c.customer_state THEN 'Same State' ELSE 'Cross State' END AS shipping_type
FROM items i
JOIN orders o ON i.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON i.seller_id = s.seller_id
GROUP BY s.seller_state, c.customer_state
ORDER BY total_orders DESC
LIMIT 15
""")

In [ ]:
# Q25: Product weight impact on freight and delivery — does heavier = slower + costlier?
sql("""
SELECT 
    CASE 
        WHEN p.product_weight_g < 500 THEN '1. Light (<500g)'
        WHEN p.product_weight_g < 2000 THEN '2. Medium (500g-2kg)'
        WHEN p.product_weight_g < 10000 THEN '3. Heavy (2-10kg)'
        ELSE '4. Very Heavy (10kg+)'
    END AS weight_category,
    COUNT(*) AS total_items,
    ROUND(AVG(i.price), 2) AS avg_price,
    ROUND(AVG(i.freight_value), 2) AS avg_freight,
    ROUND(AVG(o.actual_delivery_days), 1) AS avg_delivery_days,
    ROUND(AVG(o.delivery_delay_days), 2) AS avg_delay_days
FROM items i
JOIN products p ON i.product_id = p.product_id
JOIN orders o ON i.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY weight_category
ORDER BY weight_category
""")

## SQL Skills Demonstrated

| Skill | Queries |
|---|---|
| **Basic Aggregations** (COUNT, SUM, AVG, ROUND) | Q1, Q2, Q3, Q4 |
| **Multi-table JOINs** (INNER, LEFT — up to 5 tables) | Q5, Q6 |
| **Window Functions** (RANK, ROW_NUMBER, LAG, Running SUM) | Q7, Q8, Q9, Q10 |
| **Common Table Expressions (CTEs)** | Q11, Q12, Q13 |
| **Subqueries** (scalar, correlated, in WHERE/HAVING) | Q14, Q15 |
| **CASE Statements** (conditional logic, bucketing) | Q16, Q17, Q25 |
| **Date Functions** (strftime, julianday, date arithmetic) | Q18, Q19, Q20 |
| **HAVING clause** (post-aggregation filtering) | Q6, Q15, Q22 |
| **PARTITION BY** (window functions scoped per group) | Q9 |
| **Business Analysis** (RFM, churn, Pareto, cohort) | Q11, Q21, Q22, Q23, Q24 |

**Total: 25 queries covering all major SQL concepts tested in DA interviews.**

In [ ]:
conn.close()
print("Database connection closed.")